# 🚀 MEM LLM Orchestrator — Live Interactive Demo
### Autonomous Adaptive GPU Orchestration, Dynamic Lane Switching & Zero-OOM Defense

Welcome to the interactive testbench for **MEM Orchestrator**!
This notebook lets you experience the autonomous memory governor and adaptive lane switching engine in action on Google Colab cloud GPUs (T4 / L4 / A100) or CPU.

**What you will see in this demo:**
1. **Dynamic Hardware Calibration:** Auto-detects available GPU VRAM and tunes throughput lanes.
2. **Adaptive Lane Switching:** Watches the model promote/demote batch size and gradient accumulation in real-time.
3. **Zero-OOM Chaos Resilience:** Injects synthetic VRAM shocks and watches the MEM Governor prevent out-of-memory crashes on the fly.
4. **Telemetry & Inference:** Real-time metrics and text generation.

In [ ]:
#@title 1. Setup & Hardware Discovery
import os, sys, torch

print("=" * 65)
print("  MEM ORCHESTRATOR - HARDWARE DISCOVERY")
print("=" * 65)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  [OK] GPU Detected: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("  [!] Running on CPU (Switch to GPU in Runtime > Change runtime type)")
print("=" * 65)

# Clone repository & install dependencies
!git clone https://github.com/nobazzy/mem-llm-orchestrator.git 2>/dev/null || (cd mem-llm-orchestrator && git pull)
%cd /content/mem-llm-orchestrator/mem_v3
!pip install -q transformers datasets accelerate truststore urllib3 psutil
print("\n[OK] Dependencies installed and environment ready!")

In [ ]:
#@title 2. Run Autonomous Training Demo with Zero-OOM Defense
import os

# Execute the orchestrator demo with chaos shocks enabled
!python run_demo.py --steps 100 --model-preset medium_75m --shock-interval 30 --no-browser

In [ ]:
#@title 3. Inspect Telemetry & Memory Governance Logs
import json, glob
from pathlib import Path

log_files = glob.glob("telemetry*.json") + glob.glob("logs/*.json")
if log_files:
    print(f"Found telemetry logs: {log_files}")
    with open(log_files[0], 'r') as f:
        data = json.load(f)
        print(json.dumps(data, indent=2)[:500] + "...")
else:
    print("Demo execution completed successfully.")

In [ ]:
#@title 4. Autoregressive Text Generation
import os

prompt = "Once upon a time in a futuristic city," #@param {type:"string"}
temperature = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
max_tokens = 80 #@param {type:"integer"}

!python -u scripts/run_inference.py --prompt "{prompt}" --temperature {temperature} --max-tokens {max_tokens}